# Sample experiment: version 3 vs submission 2 on the same sample

Runs the pipeline on a sample of the **train** data and compares the current version with the
submission-2 setup on exactly the same records:

| Step | Notebook | What |
|---|---|---|
| 1 | `01_eda_preprocessing.ipynb` | linked sample: `SAMPLE_SIZE` random train S1 per country, all their true S2/S3 matches, other records in the full data's ratio |
| 2 | `02_full_e5_buckets.ipynb` | e5 buckets + extra searches, word map on (version 3) |
| 2b | `02_full_e5_buckets.ipynb` | submission 2's buckets: no word map, only the reverse / address / name searches, when `BASELINE_OWN_BUCKETS` |
| 3 | `03_full_lightgbm_submission.ipynb` | version 3 matcher (82 features, word map), cross-validation + country transfer |
| 4 | `03_full_lightgbm_submission.ipynb` | submission-2 matcher (61 features, no word map) on the step-2b buckets (or step 2's) |
| 5 | here | side-by-side table |

Every step runs in its own process (memory is freed between steps), with its output shown below as it
runs. A step whose output already exists is skipped, so after an interruption, Run all continues.

**Not a submission.** Only train is processed and 03 runs with `PREDICT_TEST=false`. A sample has fewer
lookalikes per S1 than the full data, so all scores are higher than at full size: compare the two
versions with each other, not with the leaderboard.

**Kaggle:** accelerator GPU T4 x2, Internet on (e5 model and pip packages). Inputs: the competition
data, and the repo notebooks (upload the repo folder, or at least the three notebooks, as a dataset).

**Time** (estimate, `SAMPLE_SIZE = "500k"`, `TRAIN_S1_PER_COUNTRY = 200000`): 01 ~40 min, 02 ~50 min
(x2 with `BASELINE_OWN_BUCKETS`), 03 ~1.5-2 h (x2). About 5-7 hours in total; `TRAIN_S1_PER_COUNTRY =
100000` roughly halves the two 03 steps.

In [ ]:
import os

# ---- Settings ---------------------------------------------------------------------------------------------
# (edit the defaults here; each can also be set as an environment variable of the same name)
SAMPLE_SIZE = os.environ.get("SAMPLE_SIZE", "500k")              # train S1 per country (S2/S3 follow in the full data's ratio): 100k | 250k | 500k | ...
TRAIN_S1_PER_COUNTRY = int(os.environ.get("TRAIN_S1_PER_COUNTRY", 200_000))    # S1 per country that 03 learns from (at most the sample size)
BASELINE_OWN_BUCKETS = os.environ.get("BASELINE_OWN_BUCKETS", "true") == "true"       # True: the baseline gets its own buckets without the word map (+1 run of 02)
E5_MODEL = os.environ.get("E5_MODEL", "small")                # small | base
DATA_ROOT = os.environ.get("DATA_ROOT", "")                    # folder with the competition TSVs; empty = found under /kaggle/input
REPO_DIR = os.environ.get("REPO_DIR", "")                     # folder with the notebooks; empty = found under /kaggle/input or here
# -----------------------------------------------------------------------------------------------------------
import glob
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd


def find_dir(pattern, roots):
    for root in roots:
        if root and os.path.isdir(root):
            hits = sorted((h for h in glob.glob(os.path.join(root, "**", pattern), recursive=True) if "__MACOSX" not in h),
                          key=len)
            if hits:
                return Path(hits[0]).parent
    return None


KAGGLE = os.path.isdir("/kaggle/working")
REPO = Path(REPO_DIR).resolve() if REPO_DIR else find_dir("03_full_lightgbm_submission.ipynb", [str(Path.cwd()), "/kaggle/input"])
DATA = Path(DATA_ROOT) if DATA_ROOT else find_dir("train_source1.tsv", ["/kaggle/input", str(Path.cwd().parent / "Data")])
assert REPO is not None, "notebooks not found: attach the repo (01/02/03 notebooks) as a dataset or set REPO_DIR"
assert DATA is not None, "competition data not found: attach it or set DATA_ROOT"
TAG = f"sample_{SAMPLE_SIZE}_{E5_MODEL}"
WORK = (Path("/kaggle/working" if KAGGLE else "experiments") / TAG).resolve()   # absolute: each step runs inside WORK
WORK.mkdir(parents=True, exist_ok=True)
MODEL_DIR = {"small": "multilingual-e5-small", "base": "multilingual-e5-base"}.get(E5_MODEL, E5_MODEL)
print(f"notebooks: {REPO}\ndata: {DATA}\noutput: {WORK}")

RUNNER = WORK / "_run_notebook.py"
RUNNER.write_text('''import sys, nbformat
from nbclient import NotebookClient
nb = nbformat.read(sys.argv[1], as_version=4)
def show(cell, cell_index, execute_reply, **kw):
    for o in cell.get("outputs", []):
        if o.output_type == "stream":
            print(o.text, end="", flush=True)
        elif o.output_type in ("execute_result", "display_data"):
            print(o.get("data", {}).get("text/plain", ""), flush=True)
        elif o.output_type == "error":
            print("ERROR in cell", cell_index, o.ename, o.evalue, flush=True)
client = NotebookClient(nb, timeout=None, kernel_name="python3", resources={"metadata": {"path": sys.argv[3]}},
                        on_cell_executed=show)
try:
    client.execute()
finally:
    nbformat.write(nb, sys.argv[2])
''', encoding="utf-8")


def run(step, notebook, done_file, **env):
    """Execute a notebook in its own process with these environment variables; skipped when done_file exists."""
    if Path(done_file).exists():
        print(f"== {step}: already done ({done_file}), skipped")
        return
    t0 = time.time()
    print(f"== {step}: {notebook} {env}", flush=True)
    full_env = dict(os.environ, **{k: str(v) for k, v in env.items()})
    proc = subprocess.Popen([sys.executable, str(RUNNER), str(REPO / notebook), str(WORK / f"{step}.ipynb"), str(WORK)],
                            env=full_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                            errors="replace")
    for line in proc.stdout:
        if "warning generated" not in line and "Fatal] CUDA" not in line and "recompile with CMake" not in line:
            print(line, end="")
    if proc.wait() != 0:
        failed = WORK / f"{step}.ipynb"
        try:                                                   # show the error of the failing cell
            import re as _re
            import nbformat as _nbf
            for i, cell in enumerate(_nbf.read(failed, as_version=4).cells):
                for o in cell.get("outputs", []):
                    if o.get("output_type") == "error":
                        tb = _re.sub(r"\x1b\[[0-9;]*m", "", "\n".join(o.get("traceback", [])))
                        print(f"\n---- {step}: error in cell {i}: {o.get('ename')}: {o.get('evalue')}\n{tb[-3000:]}")
        except Exception as err:
            print("could not read", failed, err)
        raise RuntimeError(f"{step} failed (error above); executed notebook: {failed}")
    print(f"== {step}: done in {(time.time() - t0) / 60:.1f} min", flush=True)

## 1 · Linked sample of the train data (01)

In [ ]:
PROCESSED = WORK / "processed"
run("step1_preprocess", "01_eda_preprocessing.ipynb", PROCESSED / "processed_counts.json",
    DATA_ROOT=DATA, FULL_RUN="false", SAMPLE_SIZE=SAMPLE_SIZE, SAMPLE_MODE="linked", PROCESS_SPLITS="train",
    PROCESSED_OUT=PROCESSED)

## 2 · Buckets, version 3 (02, word map on)

In [ ]:
EMB_V3 = WORK / "embeddings_v3"
run("step2_buckets_v3", "02_full_e5_buckets.ipynb", EMB_V3 / MODEL_DIR / "bucket_recall_train.csv",
    DATA_ROOT=DATA, PROCESSED_DIR=PROCESSED, EMB_FULL_DIR=EMB_V3, SPLITS="train", E5_MODEL=E5_MODEL, WORD_MAP="true")
EMB_BASE = WORK / "embeddings_nomap"
if BASELINE_OWN_BUCKETS:
    run("step2b_buckets_nomap", "02_full_e5_buckets.ipynb", EMB_BASE / MODEL_DIR / "bucket_recall_train.csv",
        DATA_ROOT=DATA, PROCESSED_DIR=PROCESSED, EMB_FULL_DIR=EMB_BASE, SPLITS="train", E5_MODEL=E5_MODEL, WORD_MAP="false",
        EXTRA_SOURCES="reverse,address,name")                  # submission 2's searches (no number / empty-address search)
else:
    EMB_BASE = EMB_V3

## 3 · Matcher, version 3 (03: 82 features, word map)

In [ ]:
OUT_V3 = WORK / "matcher_v3"
run("step3_matcher_v3", "03_full_lightgbm_submission.ipynb", OUT_V3 / "model" / "matcher_config.json",
    DATA_ROOT=DATA, PROCESSED_DIR=PROCESSED, BUCKET_DIR=EMB_V3 / MODEL_DIR / "buckets", SUBMISSION_DIR=OUT_V3,
    PREDICT_TEST="false", TRAIN_S1_PER_COUNTRY=TRAIN_S1_PER_COUNTRY, FEATURE_SET="all", WORD_MAP="true")

## 4 · Matcher, submission-2 setup (03: 61 features, no word map)

In [ ]:
OUT_S2 = WORK / "matcher_submission2"
run("step4_matcher_submission2", "03_full_lightgbm_submission.ipynb", OUT_S2 / "model" / "matcher_config.json",
    DATA_ROOT=DATA, PROCESSED_DIR=PROCESSED, BUCKET_DIR=EMB_BASE / MODEL_DIR / "buckets", SUBMISSION_DIR=OUT_S2,
    PREDICT_TEST="false", TRAIN_S1_PER_COUNTRY=TRAIN_S1_PER_COUNTRY, FEATURE_SET="submission2", WORD_MAP="false")

## 5 · Comparison
* **Buckets:** share of true matches inside the buckets (all S1 of the sample), overall and for
  Indian-script / Latin-script candidates, and the best score a perfect matcher could reach.
* **Matcher:** cross-validated challenge score (p ≥ t + one owner), per country, and the country
  transfer (train on one country, score the other: the stand-in for France).

In [ ]:
rows = []
for label, emb in (("version 3 (word map)", EMB_V3), ("submission 2 (no word map)", EMB_BASE)):
    r = pd.read_csv(emb / MODEL_DIR / "bucket_recall_train.csv")
    r = r[r["candidates"].str.contains("all extra") | (r["candidates"] == "e5 top-30")]
    rows.append(r.assign(buckets=label))
    if emb == EMB_BASE and not BASELINE_OWN_BUCKETS:
        break
cols = [c for c in ("buckets", "country", "candidates", "pairs per S1", "true matches in bucket", "... Indian-script candidates",
                    "... Latin-script candidates", "ceiling macro F0.5") if c in rows[0].columns]
print("BUCKETS")
display(pd.concat(rows)[cols].drop_duplicates(subset=cols[1:] if len(rows) == 1 else None).reset_index(drop=True))

rows = []
for label, out in (("version 3", OUT_V3), ("submission 2", OUT_S2)):
    cfg = json.loads((out / "model" / "matcher_config.json").read_text(encoding="utf-8"))
    m = pd.DataFrame(cfg["cv_scores"]).set_index("method")
    best = m.loc[[i for i in m.index if i.startswith("LightGBM p ≥ t + one owner")][0]]
    row = {"matcher": label, "features": len(cfg["features"]), "t": cfg["threshold"],
           **{k: round(v, 4) for k, v in best.items() if k.startswith("macro F0.5")}}
    for t in cfg.get("transfer", []):
        row[f"{t['trained on']} -> {t['scored on']}"] = round(t["macro F0.5"], 4)
    rows.append(row)
print("MATCHER (cross-validation on the sample)")
display(pd.DataFrame(rows))